In [2]:
!pip install --no-cache-dir "qiskit==2.5.2" "qiskit-machine-learning==0.9.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 189.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.3/282.3 kB 251.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 276.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 193.0 MB/s eta 0:00:00


In [3]:
!pip install --no-cache-dir "qiskit-machine-learning[torch]==0.9.1"

In [4]:
# ============================================================
# IMPORTS
# ============================================================

import sys
import getopt
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix
)

# ============================================================
# QISKIT
# ============================================================

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

# Qiskit 2.x functional API
from qiskit.circuit.library import (
    pauli_feature_map,
    real_amplitudes
)

from qiskit.primitives import StatevectorSampler as Sampler

from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")


# ============================================================
# GLOBALS
# ============================================================

root_folder = "QNNC_hybrid"


# ============================================================
# REPRODUCIBILITY
# ============================================================

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# ============================================================
# FEATURE / QUBIT SETTINGS
# ============================================================

NUM_FEATURES = 3
NUM_QUBITS = NUM_FEATURES
NUM_TARGETS = 1


# ============================================================
# QUANTUM CIRCUIT SETTINGS
# ============================================================

FEATURE_MAP_REPS_LIST = [1]

ANSATZ_REPS_LIST = [1]

ENTANGLEMENT_LIST = [
    "linear"
]


# ============================================================
# TRAINING SETTINGS
# ============================================================

LEARNING_RATE = 0.01

# Only one sample
BATCH_SIZE = 1

# Only one epoch
NUM_EPOCHS = 1


# ============================================================
# QUICK TEST SETTINGS
# ============================================================

# Only ONE training sample
TRAINING_SIZE = 1

# Only ONE test sample
TESTING_SIZE = 1


# ============================================================
# SAMPLER
# ============================================================

sampler = Sampler()


# ============================================================
# QNN MODEL
# ============================================================

def get_qnn_torch_model(
    entangle,
    feature_map_reps,
    ansatz_reps
):

    print("\n------------------------------------------")
    print("Building QNN")
    print(f"Entanglement      : {entangle}")
    print(f"Feature map reps  : {feature_map_reps}")
    print(f"Ansatz reps       : {ansatz_reps}")
    print("------------------------------------------")


    # ========================================================
    # 1. FEATURE MAP
    # ========================================================

    input_params = ParameterVector(
        "x",
        NUM_FEATURES
    )


    feature_map_template = pauli_feature_map(
        feature_dimension=NUM_FEATURES,
        reps=feature_map_reps,
        entanglement=entangle
    )


    feature_map = (
        feature_map_template.assign_parameters(
            input_params
        )
    )


    print(
        "Assigned feature map parameters:",
        feature_map.num_parameters
    )


    # ========================================================
    # 2. ANSATZ
    # ========================================================

    ansatz_template = real_amplitudes(
        num_qubits=NUM_QUBITS,
        reps=ansatz_reps,
        entanglement=entangle
    )


    num_ansatz_params = (
        ansatz_template.num_parameters
    )


    weight_params = ParameterVector(
        "theta",
        num_ansatz_params
    )


    ansatz = (
        ansatz_template.assign_parameters(
            weight_params
        )
    )


    print(
        "Assigned ansatz parameters:",
        ansatz.num_parameters
    )


    # ========================================================
    # 3. COMBINE FEATURE MAP + ANSATZ
    # ========================================================

    qc = QuantumCircuit(
        NUM_QUBITS
    )


    # Feature map
    qc.compose(
        feature_map,
        inplace=True
    )


    # Ansatz
    qc.compose(
        ansatz,
        inplace=True
    )


    print(
        "Quantum circuit created."
    )


    # ========================================================
    # 4. PARITY
    # ========================================================

    parity = lambda x: bin(x).count("1") % 2

    output_shape = 2


    # ========================================================
    # 5. SAMPLER QNN
    # ========================================================

    print(
        "Creating SamplerQNN..."
    )


    qnn = SamplerQNN(
        circuit=qc,

        input_params=input_params,

        weight_params=weight_params,

        interpret=parity,

        output_shape=output_shape,

        sampler=sampler,

        sparse=False,

        input_gradients=False
    )


    print(
        "QNN created successfully."
    )

    print(
        "Number of trainable weights:",
        qnn.num_weights
    )


    # ========================================================
    # 6. INITIAL WEIGHTS
    # ========================================================

    initial_weights = (
        0.01 *
        (
            2 *
            np.random.rand(
                qnn.num_weights
            )
            - 1
        )
    )


    # ========================================================
    # 7. TORCH CONNECTOR
    # ========================================================

    qnn_torch_model = TorchConnector(
        qnn,

        initial_weights=torch.tensor(
            initial_weights,
            dtype=torch.float32
        )
    )


    print(
        "TorchConnector created successfully."
    )


    return qnn_torch_model.to(device)


# ============================================================
# HYBRID MODEL
# ============================================================

class HybridModel(nn.Module):

    def __init__(
        self,
        qnn_model
    ):

        super().__init__()

        self.qnn = qnn_model


    def forward(self, x):

        return self.qnn(x)


# ============================================================
# DATASET PREPARATION
# ============================================================

def prepare_dataset(
    X,
    y
):

    # --------------------------------------------------------
    # First sample = training
    # Second sample = testing
    # --------------------------------------------------------

    X_train = X[
        :TRAINING_SIZE
    ]

    y_train = y[
        :TRAINING_SIZE
    ]


    X_test = X[
        TRAINING_SIZE:
        TRAINING_SIZE + TESTING_SIZE
    ]

    y_test = y[
        TRAINING_SIZE:
        TRAINING_SIZE + TESTING_SIZE
    ]


    # --------------------------------------------------------
    # Scaling
    # --------------------------------------------------------

    full_X = np.vstack([
        X_train,
        X_test
    ])


    scaler = MinMaxScaler(
        feature_range=(-1, 1)
    )


    scaler.fit(
        full_X
    )


    X_train_scaled = scaler.transform(
        X_train
    )


    X_test_scaled = scaler.transform(
        X_test
    )


    return (
        X_train_scaled,
        y_train,
        X_test_scaled,
        y_test
    )


# ============================================================
# CREATE DIRECTORIES
# ============================================================

if not os.path.exists(
    f"{root_folder}/result"
):

    os.makedirs(
        f"{root_folder}/result"
    )


if not os.path.exists(
    f"{root_folder}/logs"
):

    os.makedirs(
        f"{root_folder}/logs"
    )


# ============================================================
# DATE
# ============================================================

date = "06_08_25_1"


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print(
    f"\nFEATURE_MAP_REPS_LIST="
    f"{FEATURE_MAP_REPS_LIST}"
)

print(
    f"ANSATZ_REPS_LIST="
    f"{ANSATZ_REPS_LIST}"
)

print(
    f"ENTANGLEMENT_LIST="
    f"{ENTANGLEMENT_LIST}"
)

print(
    f"TRAINING_SIZE="
    f"{TRAINING_SIZE}"
)

print(
    f"TESTING_SIZE="
    f"{TESTING_SIZE}"
)

print(
    f"NUM_EPOCHS="
    f"{NUM_EPOCHS}"
)


# ============================================================
# LOAD DATA
# ============================================================

print(
    "\n--- Loading and Preprocessing Data ---"
)


train_df = pd.read_csv(
    "Training_Top3Features.csv"
)


test_df = pd.read_csv(
    "Testing_Top3Features.csv"
)


# ============================================================
# TRAINING DATA
# ============================================================

X_train_original = (
    train_df
    .drop(
        "went_on_backorder",
        axis=1
    )
    .values
)


y_train_original = (
    train_df[
        "went_on_backorder"
    ]
    .values
)


# ============================================================
# TEST DATA
# ============================================================

X_test_original = (
    test_df
    .drop(
        "went_on_backorder",
        axis=1
    )
    .values
)


y_test_original = (
    test_df[
        "went_on_backorder"
    ]
    .values
)


# ============================================================
# COMBINE DATA
# ============================================================

X = np.vstack([
    X_train_original,
    X_test_original
])


y = np.hstack([
    y_train_original,
    y_test_original
])


print(
    "Total number of data:",
    X.shape[0]
)


# ============================================================
# CHECK DATA SIZE
# ============================================================

if X.shape[0] < 2:

    raise ValueError(
        "Dataset must contain at least 2 samples."
    )


# ============================================================
# PREPARE ONLY 1 TRAIN + 1 TEST SAMPLE
# ============================================================

(
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test
) = prepare_dataset(
    X,
    y
)


print(
    "\nTraining data shape:",
    X_train_scaled.shape
)

print(
    "Testing data shape:",
    X_test_scaled.shape
)


# ============================================================
# CONVERT TO TORCH
# ============================================================

X_train_t = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
).to(device)


y_train_t = torch.tensor(
    y_train,
    dtype=torch.long
).to(device)


X_test_t = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
).to(device)


y_test_t = torch.tensor(
    y_test,
    dtype=torch.long
).to(device)


print(
    "X_train_t:",
    X_train_t.shape
)

print(
    "y_train_t:",
    y_train_t.shape
)

print(
    "X_test_t:",
    X_test_t.shape
)

print(
    "y_test_t:",
    y_test_t.shape
)


# ============================================================
# DATASETS
# ============================================================

train_dataset = TensorDataset(
    X_train_t,
    y_train_t
)


test_dataset = TensorDataset(
    X_test_t,
    y_test_t
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print(
    "\nNumber of training batches:",
    len(train_loader)
)

print(
    "Number of testing batches:",
    len(test_loader)
)


# ============================================================
# LOSS
# ============================================================

LOSS = nn.CrossEntropyLoss()


# ============================================================
# RESULT DATAFRAME
# ============================================================

df = pd.DataFrame(
    columns=[
        "entanglement",
        "feature_map_reps",
        "ansatz_reps",
        "actual_test",
        "predicted_test",
        "actual_train",
        "predicted_train",
        "train_accuracy",
        "test_accuracy",
        "train_f1",
        "test_f1",
        "train_recall",
        "test_recall",
        "train_precision",
        "test_precision"
    ]
)


# ============================================================
# MODEL LOOP
# ============================================================

experiment_number = 0


for entanglement in ENTANGLEMENT_LIST:

    for feature_map_reps in FEATURE_MAP_REPS_LIST:

        for ansatz_reps in ANSATZ_REPS_LIST:


            # =================================================
            # BUILD QNN
            # =================================================

            model = HybridModel(
                get_qnn_torch_model(
                    entangle=entanglement,
                    feature_map_reps=feature_map_reps,
                    ansatz_reps=ansatz_reps
                )
            ).to(device)


            # =================================================
            # OPTIMIZER
            # =================================================

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=LEARNING_RATE
            )


            # =================================================
            # TRAINING START
            # =================================================

            print(
                "\n=========================================="
            )

            print(
                f"--- Starting Training "
                f"{experiment_number}th ---"
            )

            print(
                "=========================================="
            )


            train_losses = []

            test_losses = []


            # =================================================
            # EPOCH LOOP
            # =================================================

            for epoch in range(
                NUM_EPOCHS
            ):

                model.train()

                running_loss = 0.0


                print(
                    f"\nEpoch "
                    f"{epoch + 1}/"
                    f"{NUM_EPOCHS}"
                )


                # =============================================
                # TRAINING BATCH
                # =============================================

                for batch_number, (
                    batch_X,
                    batch_y
                ) in enumerate(
                    train_loader,
                    start=1
                ):


                    print(
                        f"Processing batch "
                        f"{batch_number}/"
                        f"{len(train_loader)}..."
                    )


                    # -----------------------------------------
                    # Clear gradients
                    # -----------------------------------------

                    optimizer.zero_grad()


                    # -----------------------------------------
                    # Forward
                    # -----------------------------------------

                    outputs = model(
                        batch_X
                    )


                    print(
                        "Forward pass completed."
                    )


                    # -----------------------------------------
                    # Loss
                    # -----------------------------------------

                    loss = LOSS(
                        outputs,
                        batch_y
                    )


                    print(
                        f"Loss = {loss.item():.6f}"
                    )


                    # -----------------------------------------
                    # Backward
                    # -----------------------------------------

                    loss.backward()


                    print(
                        "Backward pass completed."
                    )


                    # -----------------------------------------
                    # Optimizer
                    # -----------------------------------------

                    optimizer.step()


                    print(
                        "Weights updated."
                    )


                    running_loss += (
                        loss.item()
                        *
                        batch_X.size(0)
                    )


                # =============================================
                # TRAIN LOSS
                # =============================================

                epoch_loss = (
                    running_loss /
                    len(train_loader.dataset)
                )


                train_losses.append(
                    epoch_loss
                )


                # =================================================
                # TEST LOSS
                # =================================================

                model.eval()

                test_loss = 0.0


                with torch.no_grad():

                    for (
                        batch_X_test,
                        batch_y_test
                    ) in test_loader:


                        outputs_test = model(
                            batch_X_test
                        )


                        loss_test = LOSS(
                            outputs_test,
                            batch_y_test
                        )


                        test_loss += (
                            loss_test.item()
                            *
                            batch_X_test.size(0)
                        )


                epoch_test_loss = (
                    test_loss /
                    len(test_loader.dataset)
                )


                test_losses.append(
                    epoch_test_loss
                )


                print(
                    f"\nEpoch "
                    f"{epoch + 1}/"
                    f"{NUM_EPOCHS} COMPLETE"
                )


                print(
                    f"Train Loss: "
                    f"{epoch_loss:.6f}"
                )


                print(
                    f"Test Loss: "
                    f"{epoch_test_loss:.6f}"
                )


            # =================================================
            # TRAINING FINISHED
            # =================================================

            print(
                "\n--- Training Finished ---"
            )


            # =================================================
            # EVALUATION
            # =================================================

            model.eval()


            all_preds = []

            all_targets = []


            all_preds_train = []

            all_targets_train = []


            with torch.no_grad():

                # ---------------------------------------------
                # TEST
                # ---------------------------------------------

                for (
                    batch_X_test,
                    batch_y_test
                ) in test_loader:


                    outputs_test = model(
                        batch_X_test
                    )


                    all_preds.extend(
                        outputs_test.cpu().numpy()
                    )


                    all_targets.extend(
                        batch_y_test.cpu().numpy()
                    )


                # ---------------------------------------------
                # TRAIN
                # ---------------------------------------------

                for (
                    batch_X_train,
                    batch_y_train
                ) in train_loader:


                    outputs_train = model(
                        batch_X_train
                    )


                    all_preds_train.extend(
                        outputs_train.cpu().numpy()
                    )


                    all_targets_train.extend(
                        batch_y_train.cpu().numpy()
                    )


            # =================================================
            # CONVERT PREDICTIONS TO LABELS
            # =================================================

            all_preds = np.array(
                all_preds
            )


            all_targets = np.array(
                all_targets
            )


            all_preds_train = np.array(
                all_preds_train
            )


            all_targets_train = np.array(
                all_targets_train
            )


            # -------------------------------------------------
            # TEST
            # -------------------------------------------------

            test_labels = []


            for item in all_preds:

                if item[1] > item[0]:

                    test_labels.append(1)

                else:

                    test_labels.append(0)


            all_preds = np.array(
                test_labels
            )


            # -------------------------------------------------
            # TRAIN
            # -------------------------------------------------

            train_labels = []


            for item in all_preds_train:

                if item[1] > item[0]:

                    train_labels.append(1)

                else:

                    train_labels.append(0)


            all_preds_train = np.array(
                train_labels
            )


            # =================================================
            # METRICS
            # =================================================

            train_accuracy = accuracy_score(
                all_targets_train,
                all_preds_train
            )


            test_accuracy = accuracy_score(
                all_targets,
                all_preds
            )


            train_f1 = f1_score(
                all_targets_train,
                all_preds_train,
                zero_division=0
            )


            test_f1 = f1_score(
                all_targets,
                all_preds,
                zero_division=0
            )


            train_recall = recall_score(
                all_targets_train,
                all_preds_train,
                zero_division=0
            )


            test_recall = recall_score(
                all_targets,
                all_preds,
                zero_division=0
            )


            train_precision = precision_score(
                all_targets_train,
                all_preds_train,
                zero_division=0
            )


            test_precision = precision_score(
                all_targets,
                all_preds,
                zero_division=0
            )


            train_cm = confusion_matrix(
                all_targets_train,
                all_preds_train
            )


            test_cm = confusion_matrix(
                all_targets,
                all_preds,
                labels=[0, 1]
            )


            # =================================================
            # PRINT RESULTS
            # =================================================

            print(
                "\n=========================================="
            )

            print(
                "--- FINAL RESULTS ---"
            )

            print(
                "=========================================="
            )


            print(
                f"Train Accuracy: "
                f"{train_accuracy:.4f}"
            )


            print(
                f"Test Accuracy: "
                f"{test_accuracy:.4f}"
            )


            print(
                f"Train F1: "
                f"{train_f1:.4f}"
            )


            print(
                f"Test F1: "
                f"{test_f1:.4f}"
            )


            print(
                f"Train Recall: "
                f"{train_recall:.4f}"
            )


            print(
                f"Test Recall: "
                f"{test_recall:.4f}"
            )


            print(
                f"Train Precision: "
                f"{train_precision:.4f}"
            )


            print(
                f"Test Precision: "
                f"{test_precision:.4f}"
            )


            print(
                "\nTrain Confusion Matrix:"
            )

            print(
                train_cm
            )


            print(
                "\nTest Confusion Matrix:"
            )

            print(
                test_cm
            )


            # =================================================
            # SAVE RESULT
            # =================================================

            new_row = {

                "entanglement":
                    entanglement,

                "feature_map_reps":
                    feature_map_reps,

                "ansatz_reps":
                    ansatz_reps,

                "actual_test":
                    all_targets.tolist(),

                "predicted_test":
                    all_preds.tolist(),

                "actual_train":
                    all_targets_train.tolist(),

                "predicted_train":
                    all_preds_train.tolist(),

                "train_accuracy":
                    train_accuracy,

                "test_accuracy":
                    test_accuracy,

                "train_f1":
                    train_f1,

                "test_f1":
                    test_f1,

                "train_recall":
                    train_recall,

                "test_recall":
                    test_recall,

                "train_precision":
                    train_precision,

                "test_precision":
                    test_precision
            }


            df.loc[
                len(df)
            ] = new_row


            # =================================================
            # SAVE CSV
            # =================================================

            file_name = (
                f"{root_folder}/result/"
                f"quick_test_"
                f"{date}.csv"
            )


            df.to_csv(
                file_name,
                index=False
            )


            print(
                f"\nResults saved to:"
                f"\n{file_name}"
            )


            experiment_number += 1


# ============================================================
# COMPLETED
# ============================================================

print(
    "\n=========================================="
)

print(
    "QUICK QNN TEST COMPLETED"
)

print(
    "=========================================="
)

Using device: cuda

FEATURE_MAP_REPS_LIST=[1]
ANSATZ_REPS_LIST=[1]
ENTANGLEMENT_LIST=['linear']
TRAINING_SIZE=1
TESTING_SIZE=1
NUM_EPOCHS=1

--- Loading and Preprocessing Data ---
Total number of data: 14000

Training data shape: (1, 3)
Testing data shape: (1, 3)
X_train_t: torch.Size([1, 3])
y_train_t: torch.Size([1])
X_test_t: torch.Size([1, 3])
y_test_t: torch.Size([1])

Number of training batches: 1
Number of testing batches: 1

------------------------------------------
Building QNN
Entanglement      : linear
Feature map reps  : 1
Ansatz reps       : 1
------------------------------------------
Assigned feature map parameters: 3
Assigned ansatz parameters: 6
Quantum circuit created.
Creating SamplerQNN...
QNN created successfully.
Number of trainable weights: 6
TorchConnector created successfully.


/usr/local/lib/python3.13/dist-packages/qiskit_machine_learning/connectors/torch_connector.py:380: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)



--- Starting Training 0th ---

Epoch 1/1
Processing batch 1/1...
Forward pass completed.
Loss = 0.691196
Backward pass completed.
Weights updated.

Epoch 1/1 COMPLETE
Train Loss: 0.691196
Test Loss: 0.695102

--- Training Finished ---

--- FINAL RESULTS ---
Train Accuracy: 0.0000
Test Accuracy: 1.0000
Train F1: 0.0000
Test F1: 0.0000
Train Recall: 0.0000
Test Recall: 0.0000
Train Precision: 0.0000
Test Precision: 0.0000

Train Confusion Matrix:
[[0 0]
 [1 0]]

Test Confusion Matrix:
[[1 0]
 [0 0]]

Results saved to:
QNNC_hybrid/result/quick_test_06_08_25_1.csv

QUICK QNN TEST COMPLETED
